In [ ]:
import zipfile
import os

zip_path = "/content/anonymisedData.zip"      # ← change this
extract_path = "./data"     # ← change this

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files:", os.listdir(extract_path))

Extracted files: ['studentInfo.csv', 'studentAssessment.csv', 'courses.csv', 'assessments.csv', 'vle.csv', 'studentVle.csv', 'studentRegistration.csv']


In [ ]:
import pandas as pd

base = "./data"  # adjust if needed

student_info = pd.read_csv(f"{base}/studentInfo.csv")
courses = pd.read_csv(f"{base}/courses.csv")
student_registration = pd.read_csv(f"{base}/studentRegistration.csv")

In [ ]:
print("student_info:", student_info.shape)
print("courses:", courses.shape)
print("student_registration:", student_registration.shape)

student_info.head()

student_info: (32593, 12)
courses: (22, 3)
student_registration: (32593, 5)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [ ]:
student_info["success"] = student_info["final_result"].isin(
    ["Pass", "Distinction"]
).astype(int)

In [ ]:
student_info["success"].value_counts()

,count
success,
0,17208
1,15385


In [ ]:
interactions = student_info[
    ["id_student", "code_module", "code_presentation", "success"]
].copy()

In [ ]:
interactions["course_id"] = (
    interactions["code_module"] + "_" + interactions["code_presentation"]
)

In [ ]:
print("Shape:", interactions.shape)
print("Students:", interactions["id_student"].nunique())
print("Courses:", interactions["course_id"].nunique())

interactions.head()

Shape: (32593, 5)
Students: 28785
Courses: 22


,id_student,code_module,code_presentation,success,course_id
0,11391,AAA,2013J,1,AAA_2013J
1,28400,AAA,2013J,1,AAA_2013J
2,30268,AAA,2013J,0,AAA_2013J
3,31604,AAA,2013J,1,AAA_2013J
4,32885,AAA,2013J,1,AAA_2013J


In [ ]:
course_popularity = interactions.groupby("course_id")["success"].sum().sort_values(ascending=False)

In [ ]:
course_popularity.head()

,success
course_id,
BBB_2014J,1152
FFF_2014J,1117
FFF_2013J,1095
BBB_2013J,1072
CCC_2014J,1015


In [ ]:
student_success = interactions[interactions["success"] == 1]

student_profile = student_success.groupby("id_student")["course_id"].apply(list)

In [ ]:
student_profile.head()

,course_id
id_student,
6516,[AAA_2014J]
11391,[AAA_2013J]
23698,[CCC_2014J]
23798,[BBB_2013J]
24186,[GGG_2014B]


In [ ]:

course_popularity.head()

,success
course_id,
BBB_2014J,1152
FFF_2014J,1117
FFF_2013J,1095
BBB_2013J,1072
CCC_2014J,1015


In [ ]:
def recommend_content(student_id, top_k=3):

    # cold start
    if student_id not in student_profile.index:
        return course_popularity.index[:top_k].tolist()

    taken = set(student_profile[student_id])

    # recommend popular unseen
    recs = [c for c in course_popularity.index if c not in taken]

    return recs[:top_k]

In [ ]:
recommend_content(6516)

['BBB_2014J', 'FFF_2014J', 'FFF_2013J']

In [ ]:
course_students = interactions.groupby("course_id")["id_student"].apply(set)

In [ ]:
len(course_students)

22

In [ ]:
def jaccard(a, b):
    return len(a & b) / len(a | b)

In [ ]:
courses_list = list(course_students.index)

jaccard(
    course_students[courses_list[0]],
    course_students[courses_list[1]]
)

0.05056179775280899

In [ ]:
def recommend_cf(student_id, top_k=3):

    # cold start
    if student_id not in student_profile.index:
        return course_popularity.index[:top_k].tolist()

    taken = set(student_profile[student_id])

    scores = {}

    for course in course_students.index:
        if course in taken:
            continue

        sim = 0
        for c in taken:
            sim += jaccard(course_students[course], course_students[c])

        scores[course] = sim

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # fallback if weak signal
    if len(ranked) == 0 or ranked[0][1] == 0:
        recs = [c for c in course_popularity.index if c not in taken]
        return recs[:top_k]

    return [c[0] for c in ranked[:top_k]]

In [ ]:
recommend_cf(6516)

['AAA_2013J', 'FFF_2013J', 'BBB_2013B']

In [ ]:
def precision_at_k_fixed(student_id, k=3):

    if student_id not in student_profile.index:
        return 0

    courses = student_profile[student_id]

    # need at least 2 courses
    if len(courses) < 2:
        return 0

    # hold one course out
    test_course = courses[-1]
    train_courses = set(courses[:-1])

    # temporarily modify profile
    original = student_profile[student_id]
    student_profile[student_id] = list(train_courses)

    preds = recommend_cf(student_id, k)

    # restore
    student_profile[student_id] = original

    return 1 if test_course in preds else 0

In [ ]:
precision_at_k(6516)

0.0

In [ ]:
import numpy as np

sample_students = list(student_profile.index[:100])

scores = [precision_at_k(s) for s in sample_students]

np.mean(scores)

np.float64(0.0)

1. Primary user and why

Primary user: Student
Reason: Recommendations directly support student course selection decisions for the next semester, making it the most immediate and impactful use case.

2. Two recommendation approaches

Option A — Content-Based

Uses student’s past successful courses
Recommends popular courses not yet taken

Option B — Collaborative Filtering

Uses interaction patterns across students
Recommends courses taken by similar students

3. Similarity metric

Jaccard similarity:

|A ∩ B| / |A ∪ B|

Used because:

Data is binary (interaction/no interaction)
Works well with sparse data
Simple and interpretable

4. Cold-start handling

For a new student:

Recommend most popular courses

This ensures valid recommendations even with no history.

5. Evaluation

Method: Leave-one-out Precision@K

One course removed from history
Model predicts it

Result:

Precision@3 ≈ 0.02

6. Comparison of approaches

Content-based → stable but not personalized
Collaborative filtering → more personalized but affected by sparse data

7. Final system

Cold-start → popularity  
Normal case → collaborative filtering  
Fallback → popularity

In [ ]:
scores = [precision_at_k_fixed(s) for s in sample_students]
np.mean(scores)

np.float64(0.02)

The observed low Precision@K (≈0.02) is primarily due to the extreme sparsity of the dataset. Most students in the OULAD dataset are enrolled in only a single course, resulting in very limited interaction history. This significantly restricts the ability of collaborative filtering methods to learn meaningful similarities between students or courses, as there is minimal overlap in behavior. Consequently, the recommendation task becomes inherently difficult, since predicting a future course based on past behavior is often not feasible. Therefore, the low precision reflects limitations in the data rather than flaws in the model design, and justifies the inclusion of a popularity-based fallback to ensure consistent recommendations.